# Phase 6B — Candidate Selection (SARIMA / LSTM / TCN)

Single Colab notebook for **validation-based configuration locking**.

## Goal
Fit candidates **A–D** on train, score on validation, lock **one config per model family** across squares [5161, 5059, 5259].

### Dec 16–22 test week is HELD OUT — not scored here.

## Protocol
- Squares / splits / L=144 from Phase 5
- Selection: mean nMAE → mean nRMSE → mean raw MAE → fit time → complexity
- Neural: train-only MinMax; SARIMA: **unscaled**, last 7 train days, maxiter=15
- Metrics: MAE, RMSE, MAPE (+ nMAE/nRMSE); Naive on validation only
- Outputs under 
esults/phase6b/ including a combined locked_configs.json

## Run order
1. Setup cells (install → Drive → imports → utils → guard)
2. **SARIMA** section (slow)
3. **LSTM** section
4. **TCN** section
5. Combine locked configs

## Google Drive layout (this project)
Raw Harvard Dataverse archives (preprocessing input — **not** used directly by Phase 6B):
- `MyDrive/dataverse_files (1).zip`
- `MyDrive/dataverse_files (2).zip`
- `MyDrive/dataverse_files (4).zip`

Windows mirror: `G:\My Drive\dataverse_files (1|2|4).zip`

Phase 6B input (Phase 5 forecasting artefacts under `milan_traffic/data/`):
- `forecasting_target_squares.csv`  (preferred)
- or `square_5161.csv` / `square_5059.csv` / `square_5259.csv`
- Results → `milan_traffic/results/phase6b/`

If `milan_traffic/data/` is empty, upload `data/processed/forecasting_target_squares.csv`
from this repo (do **not** point DATA_DIR at the raw Dataverse zips).


In [ ]:
!pip install -q statsmodels tensorflow scikit-learn pandas numpy matplotlib


## 1 · Colab setup — mount Drive & set paths


In [ ]:
# ── Google Drive mount (Colab only) ──────────────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Working folder for Phase 6B (processed square / forecasting CSVs).
# Raw Dataverse zips live beside this folder and are NOT read here:
#   /content/drive/MyDrive/dataverse_files (1).zip
#   /content/drive/MyDrive/dataverse_files (2).zip
#   /content/drive/MyDrive/dataverse_files (4).zip
# Windows mirror: G:\My Drive\dataverse_files (1|2|4).zip
if IN_COLAB:
    PROJECT_ROOT = "/content/drive/MyDrive/milan_traffic"
else:
    PROJECT_ROOT = r"G:\My Drive\milan_traffic"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS = os.path.join(PROJECT_ROOT, "results", "phase6b")
os.makedirs(RESULTS, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("RESULTS     :", RESULTS)
print(
    "data files  :",
    sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else "(missing — upload Phase 5 CSV)",
)


## 2 · Imports & protocol constants


In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Protocol constants (do not change) ───────────────────────────────────────
SQUARES   = [5161, 5059, 5259]
SEQ_LEN   = 144          # L = 1 day at 10-min resolution
HORIZON   = 1            # one-step-ahead

TRAIN_END = pd.Timestamp("2013-12-08 23:50", tz="UTC")   # inclusive
VAL_END   = pd.Timestamp("2013-12-15 23:50", tz="UTC")   # inclusive
TEST_START = pd.Timestamp("2013-12-16 00:00", tz="UTC")  # held out in Phase 6B
# TEST week Dec 16–22 is HELD OUT — not loaded or evaluated in Phase 6B


## 3 · Data utilities & metrics


In [ ]:
def _ensure_utc_index(series):
    """Make DatetimeIndex UTC-aware for comparison with protocol cutoffs."""
    idx = pd.DatetimeIndex(series.index)
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")
    out = series.copy()
    out.index = idx
    return out

def load_square(square_id, data_dir):
    """Load one square → pd.Series indexed by UTC DatetimeIndex.
    Prefers per-square CSV; falls back to forecasting_target_squares.csv.
    Raises FileNotFoundError if not found — no synthetic fallback in Phase 6B."""
    for fname in (f"square_{square_id}.csv", f"{square_id}.csv"):
        path = os.path.join(data_dir, fname)
        # Skip empty placeholder files so we fall through to the combo CSV
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            df = pd.read_csv(path, parse_dates=["timestamp"], index_col="timestamp")
            return _ensure_utc_index(df["internet_traffic"].astype(float))

    combo = os.path.join(data_dir, "forecasting_target_squares.csv")
    if os.path.isfile(combo) and os.path.getsize(combo) > 0:
        df = pd.read_csv(combo, parse_dates=["timestamp"])
        sub = df.loc[df["square_id"].astype(int) == int(square_id)].copy()
        if sub.empty:
            raise FileNotFoundError(f"square_id {square_id} missing in {combo}")
        sub = sub.sort_values("timestamp").set_index("timestamp")
        return _ensure_utc_index(sub["internet_traffic"].astype(float))

    raise FileNotFoundError(
        f"Square {square_id} not found in {data_dir}. "
        f"Expected square_{square_id}.csv or forecasting_target_squares.csv "
        f"(not the raw dataverse_files*.zip archives)"
    )

def split_series(series):
    """Return (train, val) as numpy arrays. Test window never loaded here."""
    train = series[series.index <= TRAIN_END].values
    val   = series[(series.index > TRAIN_END) & (series.index <= VAL_END)].values
    # Sanity: nothing beyond VAL_END should be loaded in Phase 6B
    assert len(train) > 0, "Train split is empty — check CSV timestamps"
    assert len(val)   > 0, "Val split is empty — check TRAIN_END/VAL_END"
    return train, val

def make_scaler(train):
    """Fit MinMaxScaler on train only; return (scaler, train_scaled)."""
    scaler = MinMaxScaler()
    tr_sc  = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    return scaler, tr_sc

def scale_val(val, scaler):
    return scaler.transform(val.reshape(-1, 1)).flatten()

def inv(arr, scaler):
    return scaler.inverse_transform(
        np.array(arr).reshape(-1, 1)).flatten()

# ── Val-sequence builder: uses trailing train history at boundary ──────────────
def make_val_sequences(train, val_scaled, scaler, seq_len=SEQ_LEN):
    """Build (X_va, y_va) for the validation window.
    The first SEQ_LEN context steps come from the END of train (unscaled →
    re-scaled), so sequences don't cold-start at the train/val boundary."""
    train_sc = scaler.transform(train.reshape(-1, 1)).flatten()
    # Concatenate last SEQ_LEN train steps + all val steps, then slice
    context  = np.concatenate([train_sc[-seq_len:], val_scaled])
    X, y = [], []
    for i in range(len(val_scaled)):
        X.append(context[i : i + seq_len])
        y.append(context[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def make_train_sequences(train_sc, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(len(train_sc) - seq_len):
        X.append(train_sc[i : i + seq_len])
        y.append(train_sc[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# ── Metrics (original scale, no R²) ──────────────────────────────────────────
def compute_metrics(y_true, y_pred, train_mean, label=""):
    """Returns MAE, RMSE, MAPE, nMAE, nRMSE (train-mean normalisation)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    # Assignment MAPE floor: max(|y|, 1e-8); do not drop near-zero rows
    denom = np.maximum(np.abs(y_true), 1e-8)
    mape = float(np.mean(np.abs(y_true - y_pred) / denom) * 100)
    nmae  = mae / train_mean     # primary selection criterion
    nrmse = rmse / train_mean    # secondary selection criterion
    if label:
        print(f"  {label:30s}  MAE={mae:,.0f}  RMSE={rmse:,.0f}  "
              f"MAPE={mape:.2f}%  nMAE={nmae:.4f}  nRMSE={nrmse:.4f}")
    return {"mae": mae, "rmse": rmse, "mape": mape, "nmae": nmae, "nrmse": nrmse}

# ── Selection rule (assignment-specified) ─────────────────────────────────────
def select_config(records):
    """
    records: list of dicts with keys:
        label, mean_nmae, mean_nrmse, mean_mae, total_fit_s, complexity
    Returns index of winning config.
    1° mean nMAE  (lower = better)
    2° mean nRMSE (lower = better)
    3° mean raw MAE
    4° total fit time
    5° complexity (number of parameters, or user-supplied int — lower = simpler)
    """
    ranked = sorted(
        range(len(records)),
        key=lambda i: (
            round(records[i]["mean_nmae"],  6),
            round(records[i]["mean_nrmse"], 6),
            round(records[i]["mean_mae"],   0),
            records[i]["total_fit_s"],
            records[i]["complexity"],
        )
    )
    return ranked[0]

# ── Naive baseline (validation) ───────────────────────────────────────────────
def naive_val_metrics(val, train_mean):
    """Persistence: ŷ(t+1) = y(t).  Operates on original-scale val array."""
    return compute_metrics(val[1:], val[:-1], train_mean, label="Naive persistence")

# ── Val plot (validation week only, NOT test) ─────────────────────────────────
def plot_val(val_true, preds_dict, square_id, model_name, results_dir=None):
    plt.figure(figsize=(14, 4))
    n = len(val_true)
    x = np.arange(n)
    plt.plot(x, val_true, label="Actual (val)", color="black", linewidth=1.2)
    colors = ["steelblue", "tomato", "seagreen", "darkorange", "purple", "brown"]
    for (lbl, pred), col in zip(preds_dict.items(), colors):
        pred = np.asarray(pred, dtype=float).reshape(-1)
        # Align lengths: pad short series at the front (e.g. Naive shift), trim long ones
        if len(pred) < n:
            pred = np.concatenate([np.full(n - len(pred), np.nan), pred])
        elif len(pred) > n:
            pred = pred[:n]
        plt.plot(x, pred, label=lbl, alpha=0.8, linewidth=1.0, color=col)
    plt.title(f"Square {square_id} — {model_name} | Validation week (Dec 9–15)")
    plt.xlabel("Step (10-min intervals)")
    plt.ylabel("Internet Traffic (bytes)")
    plt.legend(); plt.tight_layout()
    if results_dir:
        path = os.path.join(results_dir, f"sq{square_id}_{model_name.lower()}_val.png")
        plt.savefig(path, dpi=120)
        print(f"  Plot saved → {path}")
    plt.show()


## 4 · Guard: confirm test week untouched


In [ ]:
# ── Hard guard: Phase 6B working arrays must not include Dec 16–22 targets ───
# TRAIN_END / VAL_END / TEST_START are UTC-aware (defined in protocol constants).
for _sq in SQUARES:
    _s = load_square(_sq, DATA_DIR)
    _train, _val = split_series(_s)
    # Full CSV may still contain later rows; split_series must exclude them.
    _idx = _s.index
    _train_idx = _idx[_idx <= TRAIN_END]
    _val_idx = _idx[(_idx > TRAIN_END) & (_idx <= VAL_END)]
    assert len(_train_idx) == len(_train) and len(_val_idx) == len(_val)
    assert (_train_idx < TEST_START).all(), f"Square {_sq}: train overlaps test week"
    assert (_val_idx < TEST_START).all(), f"Square {_sq}: validation overlaps test week"
    assert (_val_idx <= VAL_END).all()
print("✓ Guard passed: train/val splits exclude Dec 16–22 (test may exist in CSV but is unused)")\n


## 5 · SARIMA model code

Candidates A–D from src/config.py. Unscaled traffic; last 7 train days; maxiter=15.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ── Candidate grid (from src/config.py) ──────────────────────────────────────
# No d=1, no D=1 — as specified
SARIMA_CANDIDATES = {
    "A": {"order": (1,0,1), "seasonal_order": (1,0,1,144)},
    "B": {"order": (1,0,0), "seasonal_order": (1,0,1,144)},
    "C": {"order": (0,0,1), "seasonal_order": (1,0,1,144)},
    "D": {"order": (1,0,1), "seasonal_order": (0,0,1,144)},
}

# Phase 6B fit settings (fast selection, not final training)
SARIMA_MAXITER  = 15
SARIMA_TRAIN_DAYS = 7   # use last 7 train days only for 6B selection

def sarima_fit_subset(train_unscaled, order, seasonal_order, n_days=SARIMA_TRAIN_DAYS):
    """Fit SARIMAX on UNSCALED traffic (last n_days of train only).
    Returns FilteredResults ready for .extend() rolling prediction."""
    pts = n_days * 144       # 10-min steps per day
    subset = train_unscaled[-pts:]
    p,d,q = order; P,D,Q,s = seasonal_order
    print(f"    Fitting SARIMA({p},{d},{q})({P},{D},{Q},{s}) "
          f"on {len(subset)} obs (last {n_days} train days, unscaled)...")
    t0 = time.time()
    model   = SARIMAX(subset, order=order, seasonal_order=seasonal_order,
                      enforce_stationarity=False, enforce_invertibility=False)
    res_opt = model.fit(method="lbfgs", low_memory=True,
                        maxiter=SARIMA_MAXITER, disp=False)
    res     = model.filter(res_opt.params)   # full Kalman filter for .extend()
    elapsed = time.time() - t0
    print(f"    Done in {elapsed:.1f}s | AIC={res.aic:.1f}")
    return res, elapsed

def sarima_rolling_predict(fit_result, val_unscaled):
    """One-step-ahead rolling forecast on UNSCALED val series. O(1) per step."""
    preds, current = [], fit_result
    t0 = time.time()
    for obs in val_unscaled:
        fc = current.forecast(steps=1)
        preds.append(float(np.asarray(fc).flatten()[0]))
        current = current.extend(endog=[obs])
    return np.array(preds), time.time() - t0

def sarima_complexity(order, seasonal_order):
    """Proxy complexity = total parameter count (p+q+P+Q)."""
    p,d,q = order; P,D,Q,s = seasonal_order
    return p + q + P + Q


## 6 · Run SARIMA — validation selection (no test data)

> SARIMA(s=144) is slow — budget time accordingly.


In [ ]:
# ── Phase 6B: candidate selection across all squares ─────────────────────────
# Test week (Dec 16–22) is HELD OUT and never touched here.

candidate_records = {lbl: [] for lbl in SARIMA_CANDIDATES}
per_sq_results    = {sq: {} for sq in SQUARES}

for sq in SQUARES:
    print(f"\n{'='*60}\n  Square {sq}\n{'='*60}")
    series           = load_square(sq, DATA_DIR)
    train, val       = split_series(series)
    train_mean       = train.mean()

    # Naive baseline on validation (original scale)
    print("\n  Naive baseline (validation):")
    naive_m = naive_val_metrics(val, train_mean)
    per_sq_results[sq]["naive_val"] = naive_m

    # Persistence plot series length must match val (pad with last train obs)
    naive_plot = np.concatenate([[train[-1]], val[:-1]])
    val_preds_for_plot = {"Naive": naive_plot}

    for lbl, cfg in SARIMA_CANDIDATES.items():
        print(f"\n  Candidate {lbl}:")
        try:
            res, fit_t = sarima_fit_subset(train, cfg["order"], cfg["seasonal_order"])
            preds, pred_t = sarima_rolling_predict(res, val)

            # Align: SARIMA predicts one step ahead for each val obs,
            # so preds[i] ≈ val[i] (predicting the value we then observe)
            m = compute_metrics(val, preds, train_mean,
                                label=f"SARIMA-{lbl}")
            complexity = sarima_complexity(cfg["order"], cfg["seasonal_order"])
            record = {**m, "fit_s": fit_t, "pred_s": pred_t,
                      "complexity": complexity}
            per_sq_results[sq][lbl] = record
            candidate_records[lbl].append(record)
            val_preds_for_plot[f"SARIMA-{lbl}"] = preds
        except Exception as e:
            print(f"    FAILED: {e}")
            per_sq_results[sq][lbl] = {"error": str(e)}

    # Per-square validation plot (all candidates)
    plot_val(val, val_preds_for_plot, sq, "SARIMA", RESULTS)

# ── Aggregate across squares and select winner ────────────────────────────────
print("\n" + "="*60)
print("  AGGREGATE SELECTION (mean over all 3 squares)")
print("="*60)

agg_records = []
for lbl, recs in candidate_records.items():
    valid = [r for r in recs if "error" not in r]
    if not valid:
        continue
    agg = {
        "label"      : lbl,
        "mean_nmae"  : np.mean([r["nmae"]  for r in valid]),
        "mean_nrmse" : np.mean([r["nrmse"] for r in valid]),
        "mean_mae"   : np.mean([r["mae"]   for r in valid]),
        "total_fit_s": sum(r["fit_s"]      for r in valid),
        "complexity" : valid[0]["complexity"],
    }
    agg_records.append(agg)
    cfg = SARIMA_CANDIDATES[lbl]
    print(f"  {lbl} {cfg['order']}{cfg['seasonal_order']}  "
          f"mean_nMAE={agg['mean_nmae']:.4f}  mean_nRMSE={agg['mean_nrmse']:.4f}  "
          f"mean_MAE={agg['mean_mae']:,.0f}  fit_total={agg['total_fit_s']:.0f}s")

winner_idx = select_config(agg_records)
winner_lbl = agg_records[winner_idx]["label"]
winner_cfg = SARIMA_CANDIDATES[winner_lbl]
print(f"\n  ✓ LOCKED CONFIG: SARIMA-{winner_lbl}  "
      f"order={winner_cfg['order']}  seasonal={winner_cfg['seasonal_order']}")
print("  (Full refit on train, then test evaluation: Phase 6C / later)")

# ── Save artefacts ─────────────────────────────────────────────────────────────
# 1. Per-candidate × square table
rows = []
for sq in SQUARES:
    for lbl in SARIMA_CANDIDATES:
        r = per_sq_results[sq].get(lbl, {})
        rows.append({"square": sq, "candidate": lbl, **r})
df_table = pd.DataFrame(rows)
table_path = os.path.join(RESULTS, "sarima_val_table.csv")
df_table.to_csv(table_path, index=False)
print(f"\n  Table saved → {table_path}")

# 2. Selection summary
summary = {
    "phase": "6B",
    "model": "SARIMA",
    "test_week": "Dec 16-22 — HELD OUT, not evaluated here",
    "locked_label": winner_lbl,
    "locked_order": list(winner_cfg["order"]),
    "locked_seasonal_order": list(winner_cfg["seasonal_order"]),
    "agg_records": agg_records,
}
sum_path = os.path.join(RESULTS, "sarima_selection_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"  Summary saved → {sum_path}")

# 3. Locked config JSON (machine-readable, consumed by Phase 6C)
locked = {
    "SARIMA": {
        "label": winner_lbl,
        "order": list(winner_cfg["order"]),
        "seasonal_order": list(winner_cfg["seasonal_order"]),
    }
}
locked_path = os.path.join(RESULTS, "locked_sarima_config.json")
with open(locked_path, "w") as f:
    json.dump(locked, f, indent=2)
print(f"  Locked config saved → {locked_path}")
print("\n✓ Phase 6B SARIMA complete. Test week untouched.")\n


## 7 · LSTM model code

Candidates A–D from src/config.py (includes 2-layer candidate C).


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
tf.random.set_seed(42)

# ── Candidate grid (from src/config.py) ──────────────────────────────────────
LSTM_CANDIDATES = {
    "A": {"units": 32, "n_layers": 1, "dropout": 0.1, "lr": 1e-3, "batch": 64},
    "B": {"units": 64, "n_layers": 1, "dropout": 0.1, "lr": 1e-3, "batch": 64},
    "C": {"units": 32, "n_layers": 2, "dropout": 0.1, "lr": 1e-3, "batch": 64},
    "D": {"units": 64, "n_layers": 1, "dropout": 0.2, "lr": 5e-4, "batch": 64},
}
EPOCHS   = 20   # as specified
PATIENCE = 5    # as specified

def build_lstm(units, n_layers, dropout, lr, seq_len=SEQ_LEN):
    model = Sequential()
    for i in range(n_layers):
        return_sequences = i < n_layers - 1
        if i == 0:
            model.add(LSTM(units, return_sequences=return_sequences,
                           input_shape=(seq_len, 1)))
        else:
            model.add(LSTM(units, return_sequences=return_sequences))
        model.add(Dropout(dropout))
    model.add(Dense(1))
    model.compile(optimizer=Adam(lr), loss="mse")
    return model

def lstm_train(X_tr, y_tr, X_va, y_va, cfg):
    tf.random.set_seed(42)
    model = build_lstm(cfg["units"], cfg["n_layers"], cfg["dropout"], cfg["lr"])
    cb = [
        EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    ]
    t0   = time.time()
    hist = model.fit(
        X_tr[..., np.newaxis], y_tr,
        validation_data=(X_va[..., np.newaxis], y_va),
        epochs=EPOCHS, batch_size=cfg["batch"],
        callbacks=cb, verbose=0,
    )
    elapsed = time.time() - t0
    return model, elapsed, hist

def lstm_predict(model, X, scaler):
    """Predict on scaled X, return original-scale predictions."""
    preds_sc = model.predict(X[..., np.newaxis], verbose=0).flatten()
    t0 = time.time()
    model.predict(X[:1, ..., np.newaxis], verbose=0)  # warm
    t_pred = time.time() - t0
    return inv(preds_sc, scaler), time.time() - t0

def lstm_complexity(cfg):
    """Proxy complexity: layers * units (lower = simpler for tie-break)."""
    return int(cfg["n_layers"]) * int(cfg["units"])


## 8 · Run LSTM — validation selection (no test data)


In [ ]:
# ── Phase 6B: candidate selection across all squares ─────────────────────────
# Test week (Dec 16–22) is HELD OUT and never touched here.

candidate_records = {lbl: [] for lbl in LSTM_CANDIDATES}
per_sq_results    = {sq: {} for sq in SQUARES}

for sq in SQUARES:
    print(f"\n{'='*60}\n  Square {sq}\n{'='*60}")
    series     = load_square(sq, DATA_DIR)
    train, val = split_series(series)
    scaler, tr_sc = make_scaler(train)
    va_sc         = scale_val(val, scaler)
    train_mean    = train.mean()

    # Build sequences — val uses trailing train history at boundary
    X_tr, y_tr = make_train_sequences(tr_sc)
    X_va, y_va = make_val_sequences(train, va_sc, scaler)

    # Naive baseline on validation
    print("\n  Naive baseline (validation):")
    naive_m = naive_val_metrics(val, train_mean)
    per_sq_results[sq]["naive_val"] = naive_m

    # Persistence plot series length must match val (pad with last train obs)
    naive_plot = np.concatenate([[train[-1]], val[:-1]])
    val_preds_for_plot = {"Naive": naive_plot}

    for lbl, cfg in LSTM_CANDIDATES.items():
        label_str = (f"LSTM-{lbl} units={cfg['units']} layers={cfg['n_layers']} "
                     f"drop={cfg['dropout']} lr={cfg['lr']}")
        print(f"\n  Candidate {lbl}: {label_str}")
        try:
            t0 = time.time()
            model, fit_t, hist = lstm_train(X_tr, y_tr, X_va, y_va, cfg)

            t_pred = time.time()
            preds_sc = model.predict(X_va[..., np.newaxis], verbose=0).flatten()
            pred_t   = time.time() - t_pred
            preds    = inv(preds_sc, scaler)

            m = compute_metrics(inv(y_va, scaler), preds, train_mean, label=label_str)
            complexity = lstm_complexity(cfg)
            record = {**m, "fit_s": fit_t, "pred_s": pred_t,
                      "complexity": complexity,
                      "stopped_epoch": len(hist.history["loss"])}
            per_sq_results[sq][lbl] = record
            candidate_records[lbl].append(record)
            val_preds_for_plot[f"LSTM-{lbl}"] = preds

            # Loss curve
            plt.figure(figsize=(7, 2.5))
            plt.plot(hist.history["loss"],     label="train")
            plt.plot(hist.history["val_loss"], label="val")
            plt.title(f"Sq {sq} · LSTM-{lbl} loss (stopped ep {record['stopped_epoch']})")
            plt.legend(); plt.tight_layout(); plt.show()

        except Exception as e:
            print(f"    FAILED: {e}")
            per_sq_results[sq][lbl] = {"error": str(e)}

    plot_val(val, val_preds_for_plot, sq, "LSTM", RESULTS)

# ── Aggregate across squares and select winner ────────────────────────────────
print("\n" + "="*60)
print("  AGGREGATE SELECTION (mean over all 3 squares)")
print("="*60)

agg_records = []
for lbl, recs in candidate_records.items():
    valid = [r for r in recs if "error" not in r]
    if not valid:
        continue
    agg = {
        "label"      : lbl,
        "mean_nmae"  : np.mean([r["nmae"]  for r in valid]),
        "mean_nrmse" : np.mean([r["nrmse"] for r in valid]),
        "mean_mae"   : np.mean([r["mae"]   for r in valid]),
        "total_fit_s": sum(r["fit_s"]      for r in valid),
        "complexity" : valid[0]["complexity"],
    }
    agg_records.append(agg)
    print(f"  {lbl}  mean_nMAE={agg['mean_nmae']:.4f}  mean_nRMSE={agg['mean_nrmse']:.4f}  "
          f"mean_MAE={agg['mean_mae']:,.0f}  fit_total={agg['total_fit_s']:.0f}s")

winner_idx = select_config(agg_records)
winner_lbl = agg_records[winner_idx]["label"]
winner_cfg = LSTM_CANDIDATES[winner_lbl]
print(f"\n  ✓ LOCKED CONFIG: LSTM-{winner_lbl}  {winner_cfg}")
print("  (Full refit on train, then test evaluation: Phase 6C / later)")

# ── Save artefacts ─────────────────────────────────────────────────────────────
rows = []
for sq in SQUARES:
    for lbl in LSTM_CANDIDATES:
        r = per_sq_results[sq].get(lbl, {})
        rows.append({"square": sq, "candidate": lbl, **r})
df_table = pd.DataFrame(rows)
table_path = os.path.join(RESULTS, "lstm_val_table.csv")
df_table.to_csv(table_path, index=False)
print(f"\n  Table saved → {table_path}")

summary = {
    "phase": "6B",
    "model": "LSTM",
    "test_week": "Dec 16-22 — HELD OUT, not evaluated here",
    "locked_label": winner_lbl,
    "locked_config": winner_cfg,
    "agg_records": agg_records,
}
sum_path = os.path.join(RESULTS, "lstm_selection_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"  Summary saved → {sum_path}")

locked = {"LSTM": {"label": winner_lbl, **winner_cfg}}
locked_path = os.path.join(RESULTS, "locked_lstm_config.json")
with open(locked_path, "w") as f:
    json.dump(locked, f, indent=2)
print(f"  Locked config saved → {locked_path}")
print("\n✓ Phase 6B LSTM complete. Test week untouched.")


## 9 · TCN model code

Candidates A–D from src/config.py (explicit dilations; RF = 1 + 2*(k−1)*sum(d)).


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Activation, SpatialDropout1D,
                                      Add, Dense, Lambda)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
tf.random.set_seed(42)

# ── Candidate grid (from src/config.py) ──────────────────────────────────────
TCN_CANDIDATES = {
    "A": {"filters": 32, "kernel_size": 3, "dilations": (1, 2, 4, 8),
           "dropout": 0.1, "lr": 1e-3},
    "B": {"filters": 32, "kernel_size": 3, "dilations": (1, 2, 4, 8, 16, 32),
           "dropout": 0.1, "lr": 1e-3},
    "C": {"filters": 64, "kernel_size": 3, "dilations": (1, 2, 4, 8, 16, 32),
           "dropout": 0.1, "lr": 1e-3},
    "D": {"filters": 32, "kernel_size": 5, "dilations": (1, 2, 4, 8, 16),
           "dropout": 0.1, "lr": 5e-4},
}
EPOCHS   = 20   # match LSTM protocol
PATIENCE = 5

def tcn_receptive_field(dilations, kernel_size):
    """RF = 1 + 2*(kernel_size-1)*sum(dilations); two convs per residual block."""
    return 1 + 2 * (kernel_size - 1) * int(sum(dilations))

def residual_block(x, filters, kernel_size, dilation, dropout_rate):
    """Causal dilated Conv → ReLU → SpatialDropout (×2) + residual."""
    h = Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation,
               kernel_initializer="he_normal")(x)
    h = Activation("relu")(h)
    h = SpatialDropout1D(dropout_rate)(h)
    h = Conv1D(filters, kernel_size, padding="causal", dilation_rate=dilation,
               kernel_initializer="he_normal")(h)
    h = Activation("relu")(h)
    h = SpatialDropout1D(dropout_rate)(h)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1, padding="same")(x)
    return Activation("relu")(Add()([x, h]))

def build_tcn(filters, kernel_size, dilations, dropout, lr, seq_len=SEQ_LEN):
    rf = tcn_receptive_field(dilations, kernel_size)
    print(f"    TCN RF={rf} steps (SEQ_LEN={seq_len}; wider RF not assumed better)")
    inp = Input(shape=(seq_len, 1))
    x = inp
    for d in dilations:
        x = residual_block(x, filters, kernel_size, dilation=d, dropout_rate=dropout)
    x = Lambda(lambda t: t[:, -1, :])(x)   # last timestep
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=Adam(lr), loss="mse")
    return model

def tcn_train(X_tr, y_tr, X_va, y_va, cfg):
    tf.random.set_seed(42)
    model = build_tcn(
        filters=cfg["filters"],
        kernel_size=cfg["kernel_size"],
        dilations=tuple(cfg["dilations"]),
        dropout=cfg["dropout"],
        lr=cfg["lr"],
    )
    cb = [
        EarlyStopping(patience=PATIENCE, restore_best_weights=True, monitor="val_loss"),
        ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=0),
    ]
    t0 = time.time()
    hist = model.fit(
        X_tr[..., np.newaxis], y_tr,
        validation_data=(X_va[..., np.newaxis], y_va),
        epochs=EPOCHS, batch_size=64,
        callbacks=cb, verbose=0,
    )
    return model, time.time() - t0, hist

def tcn_complexity(cfg):
    """Proxy: filters * sum(dilations) * kernel_size (lower = simpler)."""
    return int(cfg["filters"]) * int(sum(cfg["dilations"])) * int(cfg["kernel_size"])

# Print RF for each candidate at startup
print("TCN receptive fields (assignment formula):")
for lbl, cfg in TCN_CANDIDATES.items():
    rf = tcn_receptive_field(cfg["dilations"], cfg["kernel_size"])
    covers = "covers L" if rf >= SEQ_LEN else "partial L (allowed)"
    print(f"  {lbl}: RF={rf} steps ({covers})")


## 10 · Run TCN — validation selection (no test data)


In [ ]:
# ── Phase 6B: candidate selection across all squares ─────────────────────────
# Test week (Dec 16–22) is HELD OUT and never touched here.

candidate_records = {lbl: [] for lbl in TCN_CANDIDATES}
per_sq_results    = {sq: {} for sq in SQUARES}

for sq in SQUARES:
    print(f"\n{'='*60}\n  Square {sq}\n{'='*60}")
    series     = load_square(sq, DATA_DIR)
    train, val = split_series(series)
    scaler, tr_sc = make_scaler(train)
    va_sc         = scale_val(val, scaler)
    train_mean    = train.mean()

    X_tr, y_tr = make_train_sequences(tr_sc)
    X_va, y_va = make_val_sequences(train, va_sc, scaler)

    print("\n  Naive baseline (validation):")
    naive_m = naive_val_metrics(val, train_mean)
    per_sq_results[sq]["naive_val"] = naive_m

    # Persistence plot series length must match val (pad with last train obs)
    naive_plot = np.concatenate([[train[-1]], val[:-1]])
    val_preds_for_plot = {"Naive": naive_plot}

    for lbl, cfg in TCN_CANDIDATES.items():
        label_str = (f"TCN-{lbl} f={cfg['filters']} k={cfg['kernel_size']} "
                     f"dil={cfg['dilations']} drop={cfg['dropout']} lr={cfg['lr']}")
        print(f"\n  Candidate {lbl}: {label_str}")
        try:
            model, fit_t, hist = tcn_train(X_tr, y_tr, X_va, y_va, cfg)

            t0 = time.time()
            preds_sc = model.predict(X_va[..., np.newaxis], verbose=0).flatten()
            pred_t   = time.time() - t0
            preds    = inv(preds_sc, scaler)

            m = compute_metrics(inv(y_va, scaler), preds, train_mean, label=label_str)
            complexity = tcn_complexity(cfg)
            record = {**m, "fit_s": fit_t, "pred_s": pred_t,
                      "complexity": complexity,
                      "rf": tcn_receptive_field(cfg["dilations"], cfg["kernel_size"]),
                      "stopped_epoch": len(hist.history["loss"])}
            per_sq_results[sq][lbl] = record
            candidate_records[lbl].append(record)
            val_preds_for_plot[f"TCN-{lbl}"] = preds

            plt.figure(figsize=(7, 2.5))
            plt.plot(hist.history["loss"],     label="train")
            plt.plot(hist.history["val_loss"], label="val")
            plt.title(f"Sq {sq} · TCN-{lbl} loss (ep {record['stopped_epoch']})")
            plt.legend(); plt.tight_layout(); plt.show()

        except Exception as e:
            print(f"    FAILED: {e}")
            per_sq_results[sq][lbl] = {"error": str(e)}

    plot_val(val, val_preds_for_plot, sq, "TCN", RESULTS)

# ── Aggregate and select ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("  AGGREGATE SELECTION (mean over all 3 squares)")
print("="*60)

agg_records = []
for lbl, recs in candidate_records.items():
    valid = [r for r in recs if "error" not in r]
    if not valid:
        continue
    agg = {
        "label"      : lbl,
        "mean_nmae"  : np.mean([r["nmae"]  for r in valid]),
        "mean_nrmse" : np.mean([r["nrmse"] for r in valid]),
        "mean_mae"   : np.mean([r["mae"]   for r in valid]),
        "total_fit_s": sum(r["fit_s"]      for r in valid),
        "complexity" : valid[0]["complexity"],
    }
    agg_records.append(agg)
    print(f"  {lbl}  mean_nMAE={agg['mean_nmae']:.4f}  mean_nRMSE={agg['mean_nrmse']:.4f}  "
          f"mean_MAE={agg['mean_mae']:,.0f}  fit_total={agg['total_fit_s']:.0f}s")

winner_idx = select_config(agg_records)
winner_lbl = agg_records[winner_idx]["label"]
winner_cfg = TCN_CANDIDATES[winner_lbl]
print(f"\n  ✓ LOCKED CONFIG: TCN-{winner_lbl}  {winner_cfg}")
print(f"    RF = {tcn_receptive_field(winner_cfg['dilations'], winner_cfg['kernel_size'])} steps")
print("  (Full refit on train, then test evaluation: Phase 6C / later)")

# ── Save artefacts ─────────────────────────────────────────────────────────────
rows = []
for sq in SQUARES:
    for lbl in TCN_CANDIDATES:
        r = per_sq_results[sq].get(lbl, {})
        rows.append({"square": sq, "candidate": lbl, **r})
df_table = pd.DataFrame(rows)
table_path = os.path.join(RESULTS, "tcn_val_table.csv")
df_table.to_csv(table_path, index=False)
print(f"\n  Table saved → {table_path}")

summary = {
    "phase": "6B",
    "model": "TCN",
    "test_week": "Dec 16-22 — HELD OUT, not evaluated here",
    "locked_label": winner_lbl,
    "locked_config": winner_cfg,
    "locked_rf": tcn_receptive_field(winner_cfg["dilations"], winner_cfg["kernel_size"]),
    "agg_records": agg_records,
}
sum_path = os.path.join(RESULTS, "tcn_selection_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"  Summary saved → {sum_path}")

locked = {"TCN": {"label": winner_lbl,
                  "filters": winner_cfg["filters"],
                  "kernel_size": winner_cfg["kernel_size"],
                  "dilations": list(winner_cfg["dilations"]),
                  "dropout": winner_cfg["dropout"],
                  "lr": winner_cfg["lr"],
                  "rf": tcn_receptive_field(winner_cfg["dilations"], winner_cfg["kernel_size"])}}
locked_path = os.path.join(RESULTS, "locked_tcn_config.json")
with open(locked_path, "w") as f:
    json.dump(locked, f, indent=2)
print(f"  Locked config saved → {locked_path}")
print("\n✓ Phase 6B TCN complete. Test week untouched.")


## 11 · Combine locked configs

Merges per-family JSON files into locked_configs.json after all sections have run.


In [ ]:
# ── Combine locked configs from all three families ────────────────────────────
locked_combined = {'phase': '6B', 'test_week': 'Dec 16-22 — HELD OUT'}
for name in ('locked_sarima_config.json', 'locked_lstm_config.json', 'locked_tcn_config.json'):
    path = os.path.join(RESULTS, name)
    if os.path.isfile(path):
        with open(path, encoding='utf-8') as f:
            locked_combined.update(json.load(f))
        print('loaded', path)
    else:
        print('MISSING', path, '(run that model section first)')

out = os.path.join(RESULTS, 'locked_configs.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(locked_combined, f, indent=2)
print('✓ Combined locked configs →', out)
print(json.dumps(locked_combined, indent=2))
